[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-1_%EC%84%BC%EC%84%9C%EC%8B%A0%ED%98%B8%EC%99%80_%ED%8A%B9%EC%A7%95%EC%B6%94%EC%B6%9C.ipynb)

# 2-1 · 유압 장치 데이터 파악하기, 그리고 좋은 특징 고르기

**⏱ 70분** · 모듈 2: 센서 → **특징** → 이상탐지 → 고장진단 → 판정 기준

## 🎯 이번 시간의 질문

1. 이 데이터는 **무엇을, 어떻게** 기록한 것인가? (앞 30분)
2. 부품 상태를 알려면 센서의 **무엇을** 봐야 하는가? (뒤 40분)

1-1에서 분석 전에 "한 행 = 디스크 1대의 하루"부터 확인했듯이, 유압 데이터도 **어떻게 만들어졌는지** 먼저 파악합니다.

| 이번 시간 | 코드 |
|---|---|
| ✍️ **직접 입력 (8개)** — 앞으로 계속 쓰는 핵심 패턴 | 표 앞부분 `head` · 상태별 개수 `value_counts` · 배열 요약 `mean`, `axis=1` · 묶어 요약 `groupby` · 상자그림 · 구분력 계산 |
| ▶ **제공** — 한 번 보고 **결과만 읽는** 코드 | 파일 읽기, 데이터 요약, 실험 일정표, 센서 파형, 센서 범위표, 구분력 함수 |

## 0. 장치와 실험 (읽기, 5분)

독일 ZeMA 연구소의 **유압 시험장치**입니다. 공장의 유압 프레스·사출기와 같은 구성입니다(단순화한 개념도).

```
[주 회로]         전동기 ──▶ 주 펌프 ──▶ 밸브 ──▶ 부하(실린더)      ← 일을 하는 회로
                              │(내부 누설)   │(전환)
                              └──── 축압기 (압력 변동을 받아 줌)
                                     │
[냉각·여과 회로]  기름 탱크 ──▶ 여과 펌프 ──▶ 필터 ──▶ 냉각기 ──▶ 탱크   ← 기름을 식히고 거르는 회로
```

**실험 방법:** 장치는 **60초짜리 같은 작업 사이클**을 계속 반복합니다. 연구자들이 네 부품의 상태를 **일부러 나쁜 쪽으로 조정**해 가며 사이클마다 센서를 기록했고, 사이클마다 "그때 네 부품의 상태"가 정답으로 붙어 있습니다.

| 부품 | 열 이름 | 상태 값 (정상 → 고장 직전) | 나빠지면 |
|---|---|---|---|
| 냉각기 | `cooler` | 100 · 20 · 3 (%, 냉각 효율) | 기름이 과열 |
| 밸브 | `valve` | 100 · 90 · 80 · 73 (%, 전환 성능) | 유로 전환이 늦어짐 |
| 펌프 | `pump_leakage` | 0 · 1 · 2 (없음 · 약한 누설 · 심한 누설) | 내보내는 기름이 줄어듦 |
| 축압기 | `accumulator` | 130 · 115 · 100 · 90 (bar, 충전 압력) | 압력 변동을 못 받아 줌 |

| 센서 | 측정하는 것 | 단위 | 사이클(60초)당 값 |
|---|---|---|---|
| PS1 ~ PS6 | 압력 (6곳) | bar | 6,000개 (1초에 100번) |
| EPS1 | 전동기 전력 | W | 6,000개 |
| FS1, FS2 | 유량 (2곳) | L/min | 600개 (1초에 10번) |
| TS1 ~ TS4 | 기름 온도 (4곳) | °C | 60개 (1초에 1번) |
| VS1 | 진동 | mm/s | 60개 |

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "data/hydraulic_raw_4sensors.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 1. 데이터 한눈에 보기

파일은 세 개입니다: **정답**(사이클마다 네 부품 상태), **특징**(사이클마다 17개 센서를 요약한 숫자 70개), **원신호**(센서 4개의 측정값 전체).
정답과 특징을 `cycle_id`로 합쳐 `data` 한 표로 쓰고, 기본 사항을 한 번에 출력합니다.

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
data = labels.merge(features, on='cycle_id')
raw = np.load('data/hydraulic_raw_4sensors.npz')
PS1, TS1 = raw['PS1'], raw['TS1']

print('사이클 수:', len(data), '/ 열 수:', data.shape[1])
print('부품 상태 조합 수:', data['condition_group'].nunique(), '(= 3 × 4 × 3 × 4)')
print('빈칸:', data.isna().sum().sum())
print('학습용 / 평가용:', (data['split'] == 'train').sum(), '/', (data['split'] == 'test').sum())
print('조건을 바꾼 직후(stable_flag = 1):', (data['stable_flag'] == 1).sum())

**P1 · 앞부분 보기** — 몇 개 열만 골라 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data[['cycle_id', 'cooler', 'valve', 'pump_leakage', 'accumulator', 'TS1_mean']].head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — **한 행 = 장치 한 대의 60초 사이클 한 번**입니다. 2,205행은 장치 2,205대가 아니라 **한 대를 2,205번** 기록한 것입니다(1-1의 "523만 행 ≠ 523만 대"와 같은 주의).
- `cycle_id`는 시간 순서, 부품 네 열은 **정답**, `TS1_mean`(온도 TS1의 60초 평균) 같은 센서 열은 **입력**입니다.
- 위 요약: 네 부품 상태의 조합 **144가지를 전부** 시험했고, 빈칸은 없습니다. 학습용·평가용 구분과 `stable_flag`는 2절에서 설명합니다.

**P3 · 상태별 개수** — `value_counts()`를 네 부품에 반복합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for part in ['cooler', 'valve', 'pump_leakage', 'accumulator']:
    print(data[part].value_counts().sort_index())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 부품마다 **정상이 아닌 상태가 45~73%**입니다(예: 축압기 고장 직전 90 bar가 808사이클로 가장 많음). 시험장치라서 고장 데이터가 이렇게 많습니다. 현장은 고장이 드물다는 점(1-1: 0.33%)이 2-2의 출발점이 됩니다.

## 2. 실험은 어떤 순서로 진행됐나 (▶ 실험 일정표)

네 부품의 상태를 사이클 번호(= 시간) 순서로 그린 그림입니다. 코드는 실행만 하고 **그림을 읽으세요.**

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(4, 1, figsize=(11, 6), sharex=True)
for ax, (part, name) in zip(axes, [('cooler', '냉각기 (%)'), ('valve', '밸브 (%)'),
                                   ('pump_leakage', '펌프 누설'), ('accumulator', '축압기 (bar)')]):
    ax.step(data['cycle_id'], data[part], where='post')
    ax.set_ylabel(name, fontsize=9)
axes[-1].set_xlabel('사이클 번호 (시간 순서)')
fig.suptitle('실험 일정표 — 연구자가 부품 상태를 바꾼 순서')
fig.tight_layout()
plt.show()

💬 **결과 해설** — 냉각기는 3% → 20% → 100%의 **큰 세 덩어리**로, 축압기는 약 730사이클마다 한 바퀴씩, 펌프 누설은 약 40~50사이클, 밸브는 약 10사이클마다 바꿨습니다. 이 그림에서 이 데이터를 읽는 **세 가지 주의점**이 나옵니다.

1. **부품 상태가 서로 섞여 있다** — "냉각기가 정상인 사이클"에도 누설·밸브 지연·축압기 저압이 섞여 있습니다. 한 부품을 볼 때 **다른 부품의 영향이 함께 들어옵니다.**
2. **같은 조합이 연달아 기록됐다** — 이웃한 사이클은 거의 쌍둥이입니다. 무작위로 학습용/평가용을 나누면 시험 문제를 미리 본 셈이 되므로, **같은 조합은 한쪽에만** 넣었습니다(`split`: 학습 1,869 / 평가 336).
3. **상태를 바꾼 직후는 어중간하다** — 기름 온도 등이 새 상태로 옮겨 가는 중인 사이클이 756개(`stable_flag = 1`)입니다.

## 3. 원신호에서 특징으로

원신호 한 사이클이 어떻게 생겼는지 봅니다(▶ 센서 4개, 첫 사이클).

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, key, hz, unit) in zip(axes, [('PS1 압력', 'PS1', 100, 'bar'), ('FS1 유량', 'FS1', 10, 'L/min'),
                                            ('TS1 온도', 'TS1', 1, '°C'), ('VS1 진동', 'VS1', 1, 'mm/s')]):
    signal = raw[key][0]
    ax.plot(np.arange(len(signal)) / hz, signal)
    ax.set_title(f'{name} ({len(signal)}개)'); ax.set_xlabel('시간 (초)'); ax.set_ylabel(unit)
fig.tight_layout()
plt.show()

💬 압력·유량은 60초 동안 정해진 순서로 계단처럼 오르내리고, 온도는 거의 일정, 진동은 작게 흔들립니다. 압력 한 사이클에만 값이 6,000개라 그대로 비교하기 어렵습니다. 그래서 **몇 개의 숫자(특징)**로 요약합니다.

**배열 요약** — 사이클 1의 압력 6,000개를 평균·표준편차·최댓값·최솟값 네 숫자로 줄입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
x = PS1[0]
print(x.mean(), x.std(), x.max(), x.min())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 평균 160.7(전체 수준), 표준편차 13.9(흔들림 크기), 최대 191.5, 최소 145.8 bar.

`TS1.mean(axis=1)`은 **각 사이클(행)마다** 평균을 내서 2,205개를 한 번에 돌려줍니다. 제공된 `TS1_mean` 열과 같은지 확인합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(TS1.mean(axis=1)[:3])
print(data['TS1_mean'].values[:3])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 두 줄이 같으면 됩니다(35.62, 36.68, 37.88). 특징 표의 70개 열은 이렇게 **17개 센서 × 5가지 요약**(평균 `mean`, 표준편차 `std`, 최솟값 `min`, 최댓값 `max`, 마지막 1초 − 첫 1초 `change`)을 미리 계산해 둔 것입니다.

▶ 센서마다 값의 범위를 봅니다.

In [ ]:
# ▶ 그대로 실행하세요
data[['PS1_mean', 'EPS1_mean', 'FS1_mean', 'TS1_mean', 'VS1_mean']].describe().loc[['mean', 'min', 'max']].round(2)

💬 압력 약 155~181 bar, 전력 약 2,360~2,740 W, 유량 2.0~6.7 L/min, 온도 35~58°C, 진동 0.5~0.8 mm/s. **단위와 크기가 수천 배 다르므로** 센서 값을 그대로 비교하면 안 됩니다. 그래서 다음 절의 구분력처럼 **흩어짐으로 나눈 값**으로 비교합니다.

## 4. 좋은 특징이란 — 상자그림과 구분력

**P3 · 묶어 요약** — 냉각기 상태별 오일 온도 평균.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.groupby('cooler')['TS1_mean'].mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**P4 · 상자그림** — 상태별로 값의 범위(상자)와 가운데(선)를 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.boxplot(column='TS1_mean', by='cooler')
plt.suptitle('')          # pandas가 자동으로 붙이는 윗제목 지우기
plt.xlabel('냉각기 상태 (%)')
plt.ylabel('오일 온도 TS1 평균 (°C)')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기가 나빠질수록(100 → 20 → 3) 오일 온도가 약 **36 → 45 → 55°C**로 오르고 세 상자가 겹치지 않습니다. 냉각이 안 되면 기름이 뜨거워지는 물리 관계 그대로입니다. 상자 밖의 점은 2절의 "상태를 바꾼 직후" 사이클입니다.

### 구분력 — 특징이 얼마나 좋은지를 숫자 하나로

"**상자 사이의 거리가 상자 크기의 몇 배인가**"를 계산합니다.

$$\text{구분력} = \frac{\text{상태별 평균 중 가장 큰 값} - \text{가장 작은 값}}{\text{상태 안 표준편차의 평균}}$$

| 구분력 | 의미 |
|---|---|
| 3 이상 | 상자가 거의 겹치지 않음 → 특징 하나로 상태를 거의 알 수 있음 |
| 1 근처 | 상자가 많이 겹침 → 특징 하나로는 부족 |

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
g = data.groupby('cooler')['TS1_mean']
(g.mean().max() - g.mean().min()) / g.std().mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 약 **11.4**. 가장 먼 두 상태의 온도 차이(19.0°C)가 상태 안 흩어짐(평균 1.7°C)의 11배라서 상자가 겹치지 않습니다.

▶ 같은 계산을 70개 특징 전부에 반복하는 함수입니다. `rank_features(부품)` → 구분력이 높은 특징 5개.

In [ ]:
# ▶ 그대로 실행하세요
def separation(part, feature):
    """구분력 = (상태별 평균의 최댓값 − 최솟값) ÷ (상태 안 표준편차의 평균)."""
    g = data.groupby(part)[feature]
    return round((g.mean().max() - g.mean().min()) / g.std().mean(), 2)

def rank_features(part, top=5):
    """70개 센서 특징의 구분력을 높은 순서로 보여 준다."""
    cols = [c for c in features.columns if c != 'cycle_id']
    return pd.Series({c: separation(part, c) for c in cols}).sort_values(ascending=False).head(top)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
rank_features('cooler')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 냉각기에는 구분력 13 안팎의 특징이 여럿 있습니다(온도 TS4, 압력 PS5·PS6 — 기름이 뜨거워지면 묽어져 압력도 달라지기 때문으로 보임). **냉각기는 "좋은 특징"이 많은 부품**입니다. 다른 부품도 그럴까요? → 과제 2.

---
## 5. 실습 과제 (25분)

### 📝 과제 1 · 압력 PS4의 0은 무엇일까? (데이터 파악)
압력 센서 PS4의 60초 평균(`PS4_mean`)은 0인 사이클이 많습니다. **몇 개**인지 세고, **냉각기 상태와 관계가 있는지** 교차표로 확인한 뒤, 이 0을 어떻게 다뤄야 할지 적으세요.

| 냉각기 상태 | PS4 = 0인 사이클 | 전체 |
|---|---|---|
| 3% | | 732 |
| 20% | | 732 |
| 100% | | 741 |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: (data['PS4_mean'] == 0).sum()  — 조건이 참인 개수
# 힌트 2: pd.crosstab(data['cooler'], data['PS4_mean'] == 0)


✏️ **나의 답:**

### 📝 과제 2 · 부품별로 가장 좋은 특징 찾기
`rank_features()`로 나머지 세 부품의 1위 특징과 구분력을 찾아 표를 채우세요. **실행 전에** 어느 부품이 가장 어려울지 예상해 보세요.
시간이 남으면 펌프의 1위 특징을 4절처럼 상자그림으로 그려 겹침을 눈으로 확인하세요.

| 부품 | 1위 특징 | 구분력 | 특징 하나로 충분한가? |
|---|---|---|---|
| 냉각기 `cooler` | PS6_min | 13.23 | 예 |
| 펌프 `pump_leakage` | | | |
| 밸브 `valve` | | | |
| 축압기 `accumulator` | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: rank_features('pump_leakage') 처럼 부품 이름만 바꿉니다.
# 상자그림: 4절 코드에서 'TS1_mean' → 1위 특징, 'cooler' → 'pump_leakage'


✏️ **나의 답:**

### 📝 과제 3 · 밸브 특징 새로 만들기 (도전)
밸브는 1위 특징도 구분력 1.2로 약했습니다. 60초 전체를 요약한 특징이 놓친 것이 있을 수 있습니다.

1. **다른 부품은 모두 정상이고 밸브만 다른** 네 사이클의 압력을 겹쳐 그려 **차이 나는 시간대**를 찾으세요: 사이클 1788(밸브 100), 1778(90), 1768(80), 1758(73). 배열 행 번호는 0부터이므로 사이클 1788은 `PS1[1787]`, 시간 축은 `t = np.arange(6000) / 100`.
2. 찾은 구간의 압력 평균을 새 특징으로 만들고 구분력을 비교하세요. 9~10초는 `PS1[:, 900:1000]` 입니다.

| 밸브 진단용 특징 | 구분력 |
|---|---|
| FS1_max (기존 1위) | 1.18 |
| PS1_peak9 (9~10초 압력 평균) | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: plt.plot(t, PS1[1787], label='밸브 100') 을 네 번 부르고 plt.xlim(8, 12) 로 확대
# 힌트 2: data['PS1_peak9'] = PS1[:, 900:1000].mean(axis=1)  →  separation('valve', 'PS1_peak9')


✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 한 행의 의미 | 장치 1대의 60초 사이클 1번. 행 수 ≠ 설비 수 |
| 실험 설계 | 네 부품 상태를 섞은 144조합 → 다른 부품 영향이 섞인다 · 같은 조합이 연달아 → 조합 단위로 학습/평가 분리 · 상태 변경 직후는 과도 구간 |
| 센서 값 | 단위·크기가 수천 배 차이 → 흩어짐으로 나눠 비교. 빈칸이 없어도 값을 의심한다(PS4의 0) |
| 특징 | 원신호 수천 개 → 평균·표준편차 등. `배열.mean(axis=1)`로 모든 사이클 한 번에 |
| 구분력 | 상자 사이 거리 ÷ 상자 크기. 냉각기 13 · 펌프 1.1 · 밸브 1.2 · 축압기 1.2 → **냉각기만 쉬운 부품** |

다음 질문: 구분력이 높은 냉각기는 **정상 데이터만으로도** 경보를 만들 수 있을까요? 구분력이 1밖에 안 되는 펌프 누설은요? → **2-2 · 이상탐지**